# SLU Soil Moisture Classification — Clip & Resample

Processes the Sweden-wide SLU soil moisture classification map (`SLUMarkfuktighetKlassad`) for the Krycklan catchment area.

**Steps:**
1. Define spatial extent from catchment boundaries (+ 50 m buffer)
2. Clip the Sweden-wide raster to the Krycklan extent
3. Resample to target resolutions (20, 40, 80, 160 m) using the canopy height grids as reference

In [1]:
import rasterio
import numpy as np
import pandas as pd
import tools
import os
from dotenv import load_dotenv

load_dotenv()
krycklan_data_path = os.getenv('KRYCKLAN_DATA_PATH')  # root path to all GIS data
import geopandas as gpd


### 1. Define area of interest

Read the Krycklan catchments shapefile and derive a bounding box with a 50 m buffer. This `subset` is used to clip all data to the study area.

In [2]:
file = os.path.join(krycklan_data_path, 'GIS', 'CATCHMENTS', 'Krycklan_catchments', 'Krycklan catchments.shp')
df = gpd.read_file(file)

# Get bounding box of all catchments (xmin, ymin, xmax, ymax)
subset = df.total_bounds
subset = subset.round(0)

# Add 50 m buffer on all sides to avoid edge effects
subset[0] = subset[0] - 50  # xmin
subset[1] = subset[1] - 50  # ymin
subset[2] = subset[2] + 50  # xmax
subset[3] = subset[3] + 50  # ymax


### 2. Clip to Krycklan extent

Clip the Sweden-wide classified soil moisture raster to the catchment bounding box. The output is saved as an ASC file.

In [3]:
# Sweden-wide classified soil moisture raster (source data)
base_fp = os.path.join(krycklan_data_path, 'SLUMarkfuktighet/SLUMarkfuktighetskartaKlassad/SLUMarkfuktighetKlassad.tif')
# Output: clipped raster covering the Krycklan area
out_fp = os.path.join(krycklan_data_path, 'SLUMarkfuktighet/PROCESSED/SLUMarkfuktighetKlassad_clip.asc')

tools.open_raster_with_subset(base_fp, out_fp, subset, plot=False, save_in='asc')


### 3. Resample to target resolutions

Reproject and resample the clipped raster to 20, 40, 80, and 160 m resolutions using the corresponding canopy height raster as the reference grid (extent, CRS, resolution).

> **Note:** `mode` (majority) resampling is used because this is a classified (categorical) raster. For each output cell, it selects the most frequent class among all input pixels that fall within it — more representative than `nearest`, which would simply pick the class of whichever input pixel is closest to the output cell centre.

In [4]:
resos = [20, 40, 80, 160]

for reso in resos:
    # Reference grid: canopy height at target resolution defines extent, CRS and pixel size
    match = os.path.join(krycklan_data_path, f'GIS/FOREST_BASIC_KRYCKLAN/processed/canopy_height_{reso}.asc')
    # Input: clipped soil moisture raster (native resolution)
    fp = os.path.join(krycklan_data_path, 'SLUMarkfuktighet/PROCESSED/SLUMarkfuktighetKlassad_clip.asc')
    # Output: resampled to target resolution
    out_fp = os.path.join(krycklan_data_path, f'SLUMarkfuktighet/PROCESSED/SLUMarkfuktighetKlassad_clip_{reso}.asc')
    # Use mode (majority) resampling — picks the most frequent class within each output cell,
    # which is correct when aggregating a categorical map to a coarser grid
    tools.reproj_match(infile=fp, match=match, outfile=out_fp, resampling_method='mode', save_in='asc')


Coregistered to shape: 575 515 
 Affine | 20.00, 0.00, 726726.00|
| 0.00,-20.00, 7138690.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 287 257 
 Affine | 40.00, 0.00, 726726.00|
| 0.00,-40.00, 7138670.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 143 128 
 Affine | 80.00, 0.00, 726726.00|
| 0.00,-80.00, 7138630.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 71 64 
 Affine | 160.00, 0.00, 726726.00|
| 0.00,-160.00, 7138550.00|
| 0.00, 0.00, 1.00|
